# Лабораторная работа 5. Диффузионные модели (Stable Diffusion)

В этой лабораторной работе мы дообучим модель **Stable Diffusion v1.5** с помощью метода **DreamBooth** и **LoRA**, чтобы она научилась генерировать изображения с конкретным человеком (мной).

**Цели:**
1. Настроить окружение и загрузить базовую модель.
2. Сгенерировать изображения класса (Prior Preservation) для предотвращения деградации модели.
3. Обучить LoRA-адаптер на 10-15 фотографиях пользователя.
4. Сгенерировать изображения в разных стилях (киберпанк, ведьмак и т.д.).
5. Оценить качество генераций с помощью метрик (Face Similarity, CLIP Score, BRISQUE).

## §1. Dev — Подготовка окружения (Git Pull)
Эта ячейка используется в Colab для клонирования/обновления репозитория.

In [ ]:
import os
from pathlib import Path

is_colab = Path("/content").exists()
if is_colab:
    # В Colab клонируем репозиторий
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
        repo_url = f"https://{token}@github.com/Ma-XD/ITMO-CV.git"
        if not Path("/content/ITMO-CV").exists():
            !git clone {repo_url} /content/ITMO-CV
        else:
            %cd /content/ITMO-CV
            !git pull
        %cd /content/ITMO-CV
    except userdata.SecretNotFoundError:
        print("GITHUB_TOKEN не найден в Secrets. Пропускаем git pull.")
else:
    print("Локальный запуск, git pull не требуется.")

## §2. Установка зависимостей и монтирование Google Drive

In [ ]:
if is_colab:
    # Монтируем диск для доступа к фото и сохранения чекпоинтов
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Устанавливаем зависимости
    !pip install -r lab5-DIF/requirements.txt

## §3. Импорты и проверка конфигурации
Загружаем пути из `config.py` и проверяем доступность GPU.

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Добавляем папку лабы в sys.path для импорта конфигов
lab_dir = Path("lab5-DIF").resolve() if not is_colab else Path("/content/ITMO-CV/lab5-DIF")
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

from env_config import print_env, get_device
from config import (
    INSTANCE_DIR, CLASS_DIR, CHECKPOINT_DIR, FIGURE_DIR,
    MODEL_ID, INSTANCE_PROMPT, CLASS_PROMPT, RESOLUTION
)

print_env()
device = get_device()

print(f"\nПапка с фото пользователя (Instance): {INSTANCE_DIR}")
print(f"Папка с фото класса (Class): {CLASS_DIR}")
print(f"Чекпоинты сохраняются в: {CHECKPOINT_DIR}")